In [1]:
import pandas as pd
import numpy as np
import hopsworks
import os

d:\Virtual_Environments\10pearls_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv(r"..\Data\sample_data.csv")
df.head()

,time,temperature_2m,relative_humidity_2m,wind_speed_10m,surface_pressure,cloud_cover,precipitation,us_aqi,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,dust,month,hour
0,2023-08-15 00:00:00,30.7,66,7.1,984.9,0,0.0,155,75.9,212.3,1078.0,66.7,6.3,0.0,289.0,8,0
1,2023-08-15 01:00:00,30.1,67,6.1,984.4,31,0.0,155,63.8,163.9,894.0,57.3,4.8,1.0,217.0,8,1
2,2023-08-15 02:00:00,29.4,73,5.9,984.3,26,0.0,154,55.2,142.0,713.0,47.3,3.6,5.0,189.0,8,2
3,2023-08-15 03:00:00,29.1,75,4.2,984.3,28,0.0,154,53.8,163.0,557.0,37.6,3.1,11.0,254.0,8,3
4,2023-08-15 04:00:00,29.1,72,5.5,985.0,14,0.0,154,62.2,215.1,404.0,27.4,3.0,20.0,363.0,8,4


In [3]:
df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time").reset_index(drop=True)

df["hour"] = df["time"].dt.hour
df["day"] = df["time"].dt.day
df["month"] = df["time"].dt.month
df["day_of_year"] = df["time"].dt.dayofyear
df["day_of_week"] = df["time"].dt.dayofweek 

# Cyclical encoding — so Dec and Jan (both winter) end up numerically close
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

print(df[["time", "hour", "month", "month_sin", "month_cos", "hour_sin", "hour_cos"]].head())

                 time  hour  month  month_sin  month_cos  hour_sin  hour_cos
0 2023-08-15 00:00:00     0      8  -0.866025       -0.5  0.000000  1.000000
1 2023-08-15 01:00:00     1      8  -0.866025       -0.5  0.258819  0.965926
2 2023-08-15 02:00:00     2      8  -0.866025       -0.5  0.500000  0.866025
3 2023-08-15 03:00:00     3      8  -0.866025       -0.5  0.707107  0.707107
4 2023-08-15 04:00:00     4      8  -0.866025       -0.5  0.866025  0.500000


In [4]:
lag_hours = [1, 3, 6, 12, 24]

for lag in lag_hours:
    df[f"aqi_lag_{lag}h"] = df["us_aqi"].shift(lag)
    df[f"wind_speed_lag_{lag}h"] = df["wind_speed_10m"].shift(lag)
    df[f"pressure_lag_{lag}h"] = df["surface_pressure"].shift(lag)

print(df[["time", "us_aqi", "aqi_lag_1h", "aqi_lag_6h", "aqi_lag_24h"]].head(30))

                  time  us_aqi  aqi_lag_1h  aqi_lag_6h  aqi_lag_24h
0  2023-08-15 00:00:00     155         NaN         NaN          NaN
1  2023-08-15 01:00:00     155       155.0         NaN          NaN
2  2023-08-15 02:00:00     154       155.0         NaN          NaN
3  2023-08-15 03:00:00     154       154.0         NaN          NaN
4  2023-08-15 04:00:00     154       154.0         NaN          NaN
5  2023-08-15 05:00:00     154       154.0         NaN          NaN
6  2023-08-15 06:00:00     155       154.0       155.0          NaN
7  2023-08-15 07:00:00     156       155.0       155.0          NaN
8  2023-08-15 08:00:00     156       156.0       154.0          NaN
9  2023-08-15 09:00:00     157       156.0       154.0          NaN
10 2023-08-15 10:00:00     156       157.0       154.0          NaN
11 2023-08-15 11:00:00     156       156.0       154.0          NaN
12 2023-08-15 12:00:00     156       156.0       155.0          NaN
13 2023-08-15 13:00:00     155       156.0      

In [5]:
# Rolling averages — smoothed recent trend
df["aqi_roll_mean_6h"] = df["us_aqi"].rolling(window=6).mean()
df["aqi_roll_mean_24h"] = df["us_aqi"].rolling(window=24).mean()
df["aqi_roll_std_24h"] = df["us_aqi"].rolling(window=24).std()

# AQI change rate — is it rising or falling, and how fast
df["aqi_change_1h"] = df["us_aqi"].diff(1)
df["aqi_change_6h"] = df["us_aqi"].diff(6)

print(df[["time", "us_aqi", "aqi_roll_mean_6h", "aqi_roll_mean_24h", "aqi_change_1h"]].head(30))

                  time  us_aqi  aqi_roll_mean_6h  aqi_roll_mean_24h  \
0  2023-08-15 00:00:00     155               NaN                NaN   
1  2023-08-15 01:00:00     155               NaN                NaN   
2  2023-08-15 02:00:00     154               NaN                NaN   
3  2023-08-15 03:00:00     154               NaN                NaN   
4  2023-08-15 04:00:00     154               NaN                NaN   
5  2023-08-15 05:00:00     154        154.333333                NaN   
6  2023-08-15 06:00:00     155        154.333333                NaN   
7  2023-08-15 07:00:00     156        154.500000                NaN   
8  2023-08-15 08:00:00     156        154.833333                NaN   
9  2023-08-15 09:00:00     157        155.333333                NaN   
10 2023-08-15 10:00:00     156        155.666667                NaN   
11 2023-08-15 11:00:00     156        156.000000                NaN   
12 2023-08-15 12:00:00     156        156.166667                NaN   
13 202

In [6]:
print("Shape before cleanup:", df.shape)
print("Rows with any NaN:", df.isnull().any(axis=1).sum())

df_clean = df.dropna().reset_index(drop=True)

print("Shape after cleanup:", df_clean.shape)

Shape before cleanup: (26328, 44)
Rows with any NaN: 24
Shape after cleanup: (26304, 44)


In [7]:
df = df.sort_values('time').reset_index(drop=True)

In [8]:
# Multi-horizon targets for 3-day forecast (24h, 48h, 72h ahead)
df['target_aqi_24h'] = df['us_aqi'].shift(-24)
df['target_aqi_48h'] = df['us_aqi'].shift(-48)
df['target_aqi_72h'] = df['us_aqi'].shift(-72)

print(df[['time', 'us_aqi', 'target_aqi_24h', 'target_aqi_48h', 'target_aqi_72h']].tail(75))

                     time  us_aqi  target_aqi_24h  target_aqi_48h  \
26253 2026-08-12 21:00:00     154           141.0           111.0   
26254 2026-08-12 22:00:00     154           138.0           111.0   
26255 2026-08-12 23:00:00     155           135.0           111.0   
26256 2026-08-13 00:00:00     155           132.0           112.0   
26257 2026-08-13 01:00:00     155           131.0           112.0   
...                   ...     ...             ...             ...   
26323 2026-08-15 19:00:00     115             NaN             NaN   
26324 2026-08-15 20:00:00     115             NaN             NaN   
26325 2026-08-15 21:00:00     113             NaN             NaN   
26326 2026-08-15 22:00:00     112             NaN             NaN   
26327 2026-08-15 23:00:00     111             NaN             NaN   

       target_aqi_72h  
26253           113.0  
26254           112.0  
26255           111.0  
26256             NaN  
26257             NaN  
...               ...  
263

In [9]:
print(df.shape)
print(df['time'].min(), df['time'].max())

(26328, 47)
2023-08-15 00:00:00 2026-08-15 23:00:00


In [10]:
os.makedirs(r"D:\tmp", exist_ok=True)

project = hopsworks.login(
    api_key_value=os.getenv("AQI_Predictor_KEY"),
    cert_folder="./hopsworks-certs"
)
fs = project.get_feature_store()

fg_v2 = fs.get_or_create_feature_group(
    name="aqi_features_multan",
    version=2,
    primary_key=["time"],
    event_time="time",
    time_travel_format="HUDI", 
    description="AQI features for Multan with 24h/48h/72h multi-horizon targets",
    online_enabled=False,  
    stream=False,
)

2026-08-22 15:59:03,936 INFO: Initializing external client
2026-08-22 15:59:03,936 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-08-22 15:59:08,745 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/41139


In [11]:
os.makedirs(r"D:\tmp", exist_ok=True)

fg_v2.insert(df, write_options={"wait_for_job": True})

Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/41139/fs/29817/fg/50951


Uploading Dataframe: 100.00% |██████████| Rows 26328/26328 | Elapsed Time: 00:24 | Remaining Time: 00:00


Launching job: aqi_features_multan_2_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/41139/jobs/named/aqi_features_multan_2_offline_fg_materialization/executions


2026-08-22 15:59:57,858 INFO: Waiting for execution to finish. Current state: SUBMITTED. Final status: UNDEFINED
2026-08-22 16:00:01,307 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2026-08-22 16:03:33,396 INFO: Waiting for execution to finish. Current state: FAILED. Final status: FAILED
2026-08-22 16:03:34,214 INFO: Waiting for log aggregation to finish.
2026-08-22 16:03:34,217 ERROR: Execution failed with status: FAILED. See the logs for more information.


JobExecutionException: The Hopsworks Job failed, use the Hopsworks UI to access the job logs

In [13]:
fg_v2 = fs.get_feature_group(name="aqi_features_multan", version=2)
df = fg_v2.read()
df = df.sort_values('time').reset_index(drop=True)
print(df.shape)
print(df[['time', 'us_aqi', 'target_aqi_24h', 'target_aqi_48h', 'target_aqi_72h']].tail(100))

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (15.09s) 
(26328, 47)
                           time  us_aqi  target_aqi_24h  target_aqi_48h  \
26228 2026-08-11 20:00:00+00:00     137           154.0           143.0   
26229 2026-08-11 21:00:00+00:00     138           154.0           141.0   
26230 2026-08-11 22:00:00+00:00     140           154.0           138.0   
26231 2026-08-11 23:00:00+00:00     142           155.0           135.0   
26232 2026-08-12 00:00:00+00:00     143           155.0           132.0   
...                         ...     ...             ...             ...   
26323 2026-08-15 19:00:00+00:00     115             NaN             NaN   
26324 2026-08-15 20:00:00+00:00     115             NaN             NaN   
26325 2026-08-15 21:00:00+00:00     113             NaN             NaN   
26326 2026-08-15 22:00:00+00:00     112             NaN             NaN   
26327 2026-08-15 23:00:00+00:00     111             NaN             NaN   

